In [ ]:
#Overview of Student's performance in Math subject
import pandas as pd
import zipfile
with zipfile.ZipFile('/content/archive.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')
df = pd.read_csv('student_math_clean.csv')
df.head()

In [ ]:
df.info()


**Encoding**

In [ ]:
#Binary: Misc
df["school"] = df["school"].replace({"MS": 1, "GP": 0})
df["sex"] = df["sex"].replace({"F": 1, "M": 0})
df["address_type"] = df["address_type"].replace({"Urban": 1, "Rural": 0})
df["family_size"] = df["family_size"].replace({"Greater than 3": 1, "Less than or equal to 3": 0})
df["parent_status"] = df["parent_status"].replace({"Living together": 1, "Apart": 0})

#Binary: Yes/No
df["school_support"] = df["school_support"].replace({"yes": 1, "no": 0})
df["family_support"] = df["family_support"].replace({"yes": 1, "no": 0})
df["extra_paid_classes"] = df["extra_paid_classes"].replace({"yes": 1, "no": 0})
df["activities"] = df["activities"].replace({"yes": 1, "no": 0})
df["nursery_school"] = df["nursery_school"].replace({"yes": 1, "no": 0})
df["higher_ed"] = df["higher_ed"].replace({"yes": 1, "no": 0})
df["internet_access"] = df["internet_access"].replace({"yes": 1, "no": 0})
df["romantic_relationship"] = df["romantic_relationship"].replace({"yes": 1, "no": 0})

In [ ]:
#Ordinal
travel = {'<15 min.': 0, '15 to 30 min.': 1, '30 min. to 1 hour': 2, '>1 hour':3}
df["travel_time"] = df["travel_time"].map(travel)

study = {'<2 hours': 0, '2 to 5 hours': 1, '5 to 10 hours': 2, '>10 hours':3}
df["study_time"] = df["study_time"].map(study)

In [ ]:
#One-hot
categorical_cols = ["mother_education", "mother_job", "father_education", "father_job", "school_choice_reason", "guardian"]
encoded_df = pd.get_dummies(df, columns=categorical_cols, prefix=categorical_cols, prefix_sep='_')

In [ ]:
df["school_choice_reason"].value_counts()

In [ ]:
encoded_df

In [ ]:
encoded_df.info()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

correlation_matrix = encoded_df.drop(["student_id"], axis = 1).corr()  # Calculate the correlation matrix

plt.figure(figsize=(30,30))
sns.heatmap(correlation_matrix, annot=True, square=True,
            cmap='RdBu', vmax=1, vmin=-1)
plt.title('Correlations Between Variables', size=18)
plt.xticks(size=13)
plt.yticks(size=13)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Assuming 'correlation_matrix' is your correlation matrix
correlation_matrix = encoded_df.drop(["student_id"], axis=1).corr()

# Set the threshold
threshold = 0.7

# Get pairs with high correlation
highly_correlated_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > threshold:
            highly_correlated_pairs.append(
                (correlation_matrix.columns[i], correlation_matrix.columns[j], correlation_matrix.iloc[i, j])
            )

# Print the highly correlated pairs
print("Highly Correlated Pairs (above threshold of 0.7):")
for pair in highly_correlated_pairs:
    print(f"{pair[0]} and {pair[1]}: {pair[2]:.2f}")

**Get X and y**

In [ ]:
y = encoded_df["final_grade"]
X = encoded_df[["grade_2","grade_1","absences","family_relationship","age"]]

In [ ]:
y

**Train - test split**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**Linear regression model**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate MSE and R-squared for training set
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Calculate MSE and R-squared for testing set
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# Print the results
print(f"Training MSE: {train_mse:.2f}")
print(f"Training RMSE: {(train_mse)**0.5:.2f}")
print(f"Training R-squared: {train_r2:.2f}")
print(f"Testing MSE: {test_mse:.2f}")
print(f"Testing RMSE: {(test_mse)**0.5:.2f}")
print(f"Testing R-squared: {test_r2:.2f}")

In [ ]:
y_test.mean(), y_test.min(), y_test.max()

**kNN model**

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# Create and train the k-NN model
knn_model = KNeighborsRegressor(n_neighbors=5)  # You can adjust the number of neighbors
knn_model.fit(X_train, y_train)

# Make predictions
knn_y_train_pred = knn_model.predict(X_train)
knn_y_test_pred = knn_model.predict(X_test)

# Evaluate the k-NN model
knn_train_mse = mean_squared_error(y_train, knn_y_train_pred)
knn_train_r2 = r2_score(y_train, knn_y_train_pred)
knn_test_mse = mean_squared_error(y_test, knn_y_test_pred)
knn_test_r2 = r2_score(y_test, knn_y_test_pred)

print(f"k-NN Training MSE: {knn_train_mse:.2f}")
print(f"k-NN Training RMSE: {(knn_train_mse)**0.5:.2f}")
print(f"k-NN Training R-squared: {knn_train_r2:.2f}")
print(f"k-NN Testing MSE: {knn_test_mse:.2f}")
print(f"k-NN Testing RMSE: {(knn_test_mse)**0.5:.2f}")
print(f"k-NN Testing R-squared: {knn_test_r2:.2f}")

**Decision tree**

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Create and train the decision tree model
dt_model = DecisionTreeRegressor(random_state=42)  # You can adjust hyperparameters
dt_model.fit(X_train, y_train)

# Make predictions
dt_y_train_pred = dt_model.predict(X_train)
dt_y_test_pred = dt_model.predict(X_test)

# Evaluate the decision tree model
dt_train_mse = mean_squared_error(y_train, dt_y_train_pred)
dt_train_r2 = r2_score(y_train, dt_y_train_pred)
dt_test_mse = mean_squared_error(y_test, dt_y_test_pred)
dt_test_r2 = r2_score(y_test, dt_y_test_pred)

print(f"Decision Tree Training MSE: {dt_train_mse:.2f}")
print(f"Decision Tree Training RMSE: {(dt_train_mse)**0.5:.2f}")
print(f"Decision Tree Training R-squared: {dt_train_r2:.2f}")
print(f"Decision Tree Testing MSE: {dt_test_mse:.2f}")
print(f"Decision Tree Testing RMSE: {(dt_test_mse)**0.5:.2f}")
print(f"Decision Tree Testing R-squared: {dt_test_r2:.2f}")

**Random forest model**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Create and train the random forest model
rf_model = RandomForestRegressor(max_features = 49, min_samples_split = 10, n_estimators = 100)  # You can adjust hyperparameters
rf_model.fit(X_train, y_train)

# Make predictions
rf_y_train_pred = rf_model.predict(X_train)
rf_y_test_pred = rf_model.predict(X_test)

# Evaluate the random forest model
rf_train_mse = mean_squared_error(y_train, rf_y_train_pred)
rf_train_r2 = r2_score(y_train, rf_y_train_pred)
rf_test_mse = mean_squared_error(y_test, rf_y_test_pred)
rf_test_r2 = r2_score(y_test, rf_y_test_pred)
mae_train = mean_absolute_error(y_train, rf_y_train_pred)
mae_test = mean_absolute_error(y_test, rf_y_test_pred)


print(f"Random Forest Training MSE: {rf_train_mse:.2f}")
print(f"Random Forest Training RMSE: {(rf_train_mse)**0.5:.2f}")
print(f"Random Forest Training R-squared: {rf_train_r2:.2f}")
print(f"Random Forest Testing MSE: {rf_test_mse:.2f}")
print(f"Random Forest Testing RMSE: {(rf_test_mse)**0.5:.2f}")
print(f"Random Forest Testing R-squared: {rf_test_r2:.2f}")

print(f"RF Training MAE: {mae_train:.2f}")
print(f"RF Testing MAE: {mae_test:.2f}")

Hyperarameter tuning for Random forest

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the hyperparameter grid to search
param_grid = {
    'n_estimators': [500, 1000],
    #'max_depth': [None, 5, 10, 15], #Try bring this in
    'min_samples_split': range(50, 100, 10), #Change the first '10' to reduce overfitting --> Try 40, 30, (20, 60)
    'max_features': range(1, len(X.columns), 2) #Change to 1,...
}

rf_model = RandomForestRegressor(random_state=42)
# Create the GridSearchCV object
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid,
                           scoring='neg_mean_squared_error', cv= 3 , verbose=5)

# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, y_train)

# Get the best hyperparameters
best_params = grid_search.best_params_
print(f"Best Hyperparameters: {best_params}")

# Get the best model
best_rf_model = grid_search.best_estimator_

# Make predictions using the best model
y_pred_test = best_rf_model.predict(X_test)
y_pred_train = best_rf_model.predict(X_train)

# Evaluate the best model
mse_train = mean_squared_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print(f"MSE Train: {mse_train:.2f}")
print(f"RMSE Train: {(mse_train)**0.5:.2f}")
print(f"R-squared Train: {r2_train:.2f}")
print(f"MSE Test: {mse_test:.2f}")
print(f"RMSE Test: {(mse_test)**0.5:.2f}")
print(f"R-squared Test: {r2_test:.2f}")

**XGBoost model**

In [ ]:
import xgboost as xgb

# Create and train the XGBoost model
xgb_model = xgb.XGBRegressor(alpha = 5, colsample_bynode = 0.8, eta = 0.1, n_estimators = 100, subsample = 1) # You can adjust hyperparameters
xgb_model.fit(X_train, y_train)

# Make predictions
xgb_y_train_pred = xgb_model.predict(X_train)
xgb_y_test_pred = xgb_model.predict(X_test)

# Evaluate the XGBoost model
xgb_train_mse = mean_squared_error(y_train, xgb_y_train_pred)
xgb_train_r2 = r2_score(y_train, xgb_y_train_pred)
xgb_test_mse = mean_squared_error(y_test, xgb_y_test_pred)
xgb_test_r2 = r2_score(y_test, xgb_y_test_pred)
mae_train = mean_absolute_error(y_train, xgb_y_train_pred)
mae_test = mean_absolute_error(y_test, xgb_y_test_pred)

print(f"XGBoost Training MSE: {xgb_train_mse:.2f}")
print(f"XGBoost Training RMSE: {(xgb_train_mse)**0.5:.2f}")
print(f"XGBoost Training R-squared: {xgb_train_r2:.2f}")
print(f"XGBoost Testing MSE: {xgb_test_mse:.2f}")
print(f"XGBoost Testing RMSE: {(xgb_test_mse)**0.5:.2f}")
print(f"XGBoost Testing R-squared: {xgb_test_r2:.2f}")

print(f"XGBoost Training MAE: {mae_train:.2f}")
print(f"XGBoost Testing MAE: {mae_test:.2f}")

Hyperparameter tuning for XGBoost

In [ ]:
# Define the parameter grid for XGBoost
from sklearn.model_selection import GridSearchCV
!pip install --upgrade scikit-learn xgboost
# Restart the kernel after installing
param_grid_xgb = {
    'n_estimators': [100, 200],
    #'max_depth': [3, 5, 7],
    'eta': [0.01, 0.05, 0.1, 0.2, 0.3],
    'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1],
    'colsample_bynode': [0.5, 0.6, 0.7, 0.8, 0.9, 1],
    'alpha': [0, 0.1, 0.5, 1, 5, 10],
    'lambda': [0, 0.1, 0.5, 1, 5, 10]
}
# Create the XGBoost model
xgb_model = xgb.XGBRegressor(random_state=42)
# Perform GridSearchCV for hyperparameter tuning
grid_search_xgb = GridSearchCV(estimator=xgb_model, param_grid=param_grid_xgb,
                              scoring='neg_mean_squared_error', cv=2, verbose=5)
grid_search_xgb.fit(X_train, y_train)
# Print the best hyperparameters
print("Best hyperparameters:", grid_search_xgb.best_params_)
# Evaluate the best model
best_xgb_model = grid_search_xgb.best_estimator_
xgb_y_pred_train = best_xgb_model.predict(X_train)  # Predictions for training data
xgb_y_pred_test = best_xgb_model.predict(X_test)  # Predictions for testing data
xgb_mse_train = mean_squared_error(y_train, xgb_y_pred_train)
xgb_r2_train = r2_score(y_train, xgb_y_pred_train)
xgb_mse_test = mean_squared_error(y_test, xgb_y_pred_test)
xgb_r2_test = r2_score(y_test, xgb_y_pred_test)


print(f"XGBoost best model Train MSE: {xgb_mse_train:.2f}")
print(f"XGBoost best model Train RMSE: {xgb_mse_train**0.5:.2f}")
print(f"XGBoost best model Train R-squared: {xgb_r2_train:.2f}")
print(f"XGBoost best model Test MSE: {xgb_mse_test:.2f}")
print(f"XGBoost best model Test RMSE: {xgb_mse_test**0.5:.2f}")
print(f"XGBoost best model Test R-squared: {xgb_r2_test:.2f}")